In [1]:
import pandas as pd
import os
from tasks.classification import parse_class_to_dataset

In [3]:
# Path to the per_class.csv file
output_dir = "/group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/zero_shot_classification/gpt-5-nano/bean_disease_uganda"  # Change this to your output directory
per_class_csv = os.path.join(output_dir, "per_class.csv")

# Load per-class metrics
df = pd.read_csv(per_class_csv)
print(f"Loaded {len(df)} classes from {per_class_csv}")
df.head()

Loaded 3 classes from /group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/zero_shot_classification/gpt-5-nano/bean_disease_uganda/per_class.csv


,class,precision,recall,f1,support
0,angular_leaf_spot,0.533679,0.985646,0.692437,209.0
1,bean_rust,0.900000,0.041860,0.080000,215.0
2,healthy,0.870293,0.985782,0.924444,211.0


In [4]:
# Parse dataset name from class name
df['dataset'], df['class_name'] = zip(*df['class'].apply(parse_class_to_dataset))

# Show the parsed data
print(f"\nFound {df['dataset'].nunique()} unique datasets:")
print(df['dataset'].unique())
df.head(10)


Found 1 unique datasets:
['unknown']


,class,precision,recall,f1,support,dataset,class_name
0,angular_leaf_spot,0.533679,0.985646,0.692437,209.0,unknown,angular_leaf_spot
1,bean_rust,0.900000,0.041860,0.080000,215.0,unknown,bean_rust
2,healthy,0.870293,0.985782,0.924444,211.0,unknown,healthy


In [5]:
# Aggregate metrics by dataset
# We'll compute:
# - Average precision, recall, f1 across classes in each dataset
# - Total support (number of samples) per dataset
# - Number of classes per dataset
# - Accuracy per dataset (weighted by support)

# First, compute weighted accuracy per dataset
def compute_dataset_accuracy(group):
    """Compute accuracy as weighted average of per-class recall by support"""
    total_support = group['support'].sum()
    if total_support == 0:
        return 0.0
    weighted_recall = (group['recall'] * group['support']).sum() / total_support
    return weighted_recall

dataset_metrics = df.groupby('dataset').agg({
    'precision': 'mean',
    'recall': 'mean',
    'f1': 'mean',
    'support': 'sum',
    'class': 'count'  # counts number of classes
}).rename(columns={'class': 'n_classes'})

# Add accuracy as weighted average
dataset_metrics['accuracy'] = df.groupby('dataset').apply(compute_dataset_accuracy)

# Reorder columns
dataset_metrics = dataset_metrics[['n_classes', 'support', 'accuracy', 'precision', 'recall', 'f1']]

# Round to 4 decimal places
dataset_metrics = dataset_metrics.round(4)

# Sort by dataset name
dataset_metrics = dataset_metrics.sort_index()

print(f"\nDataset-level metrics:")
dataset_metrics


Dataset-level metrics:


/tmp/ipykernel_696361/2310309574.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataset_metrics['accuracy'] = df.groupby('dataset').apply(compute_dataset_accuracy)


,n_classes,support,accuracy,precision,recall,f1
dataset,,,,,,
unknown,3,635.0,0.6661,0.768,0.6711,0.5656


In [6]:
# Save to CSV
dataset_metrics_csv = os.path.join(output_dir, "dataset_metrics.csv")
dataset_metrics.to_csv(dataset_metrics_csv)

print(f"\nSaved dataset metrics to: {dataset_metrics_csv}")


Saved dataset metrics to: /group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/zero_shot_classification/gpt-5-nano/bean_disease_uganda/dataset_metrics.csv


In [7]:
# Optional: Show summary statistics
print("\nSummary Statistics Across All Datasets:")
print(f"Total datasets: {len(dataset_metrics)}")
print(f"Total samples: {dataset_metrics['support'].sum()}")
print(f"Total classes: {dataset_metrics['n_classes'].sum()}")
print(f"\nAverage metrics across datasets:")
print(f"  Accuracy: {dataset_metrics['accuracy'].mean():.4f}")
print(f"  Precision: {dataset_metrics['precision'].mean():.4f}")
print(f"  Recall: {dataset_metrics['recall'].mean():.4f}")
print(f"  F1 Score: {dataset_metrics['f1'].mean():.4f}")


Summary Statistics Across All Datasets:
Total datasets: 1
Total samples: 635.0
Total classes: 3

Average metrics across datasets:
  Accuracy: 0.6661
  Precision: 0.7680
  Recall: 0.6711
  F1 Score: 0.5656
